# Importação das Bibliotecas necessárias

In [10]:
from ultralytics import YOLO
import os
from IPython.display import display, Image
from IPython import display
import requests
display.clear_output()


# Donwload do DataSet

In [ ]:
import os
import requests
import zipfile
import shutil

# URL do arquivo zip no Google Drive (exemplo)
drive_url = "https://drive.usercontent.google.com/download?id=1VGnPFfXVZcLY2-fT3M7zXKYTPXMSAaja&export=download&authuser=0&confirm=t&uuid=56adfbe7-4b31-43b4-a026-f07601384fcb&at=AN8xHooAXPh68VgPFMRtWTtDg3Wo%3A1758289282777"

# Nome do arquivo zip local
zip_filename = "dataset.zip"

# Pasta de destino
dest_dir = "Data//fruit.v11i.yolov11"

print("🔍 VERIFICANDO DATASET EXISTENTE...")
print("=" * 50)

# Verificar se o dataset já existe
dataset_exists = False
if os.path.exists(dest_dir):
    # Verificar se contém dados válidos
    subdirs = [d for d in os.listdir(dest_dir) if os.path.isdir(os.path.join(dest_dir, d))]
    
    if subdirs:
        print(f"📁 Pasta 'Data' encontrada com {len(subdirs)} subpasta(s):")
        for subdir in subdirs:
            subdir_path = os.path.join(dest_dir, subdir)
            
            # Contar arquivos na subpasta
            total_files = 0
            for root, dirs, files in os.walk(subdir_path):
                total_files += len(files)
            
            print(f"   📂 {subdir} - {total_files} arquivos")
        
        dataset_exists = True
        
        # Verificar especificamente o dataset de frutas
        fruit_dataset_path = os.path.join(dest_dir, "fruit.v11i.yolov11")
        if os.path.exists(fruit_dataset_path):
            data_yaml = os.path.join(fruit_dataset_path, "data.yaml")
            if os.path.exists(data_yaml):
                print(f"✅ Dataset de frutas completo encontrado!")
            else:
                print(f"⚠️ Dataset de frutas incompleto (falta data.yaml)")
        else:
            print(f"❌ Dataset de frutas não encontrado")
    else:
        print(f"📁 Pasta 'Data' existe mas está vazia")
else:
    print(f"❌ Pasta 'Data' não encontrada")

# Decidir se deve fazer download
should_download = False

if dataset_exists:
    print(f"\n❓ CONFIRMAÇÃO NECESSÁRIA:")
    print(f"Um dataset já foi encontrado na pasta 'Data'.")
    print(f"")
    print(f"Opções:")
    print(f"  1️⃣ Pular download (usar dataset existente)")
    print(f"  2️⃣ Baixar novamente (sobrescrever)")
    print(f"  3️⃣ Baixar em nova pasta (manter ambos)")
    
    choice = input(f"\n👆 Digite sua escolha (1/2/3): ").strip()
    
    if choice == "1":
        print(f"✅ Usando dataset existente!")
        should_download = False
    elif choice == "2":
        print(f"⚠️ Sobrescrevendo dataset existente...")
        should_download = True
        # Limpar pasta existente
        if os.path.exists(dest_dir):
            shutil.rmtree(dest_dir)
    elif choice == "3":
        print(f"📁 Criando nova pasta para o dataset...")
        should_download = True
        dest_dir = f"Data_new_{int(time.time())}"  # Pasta única
        print(f"   Nova pasta: {dest_dir}")
    else:
        print(f"❌ Opção inválida. Cancelando download.")
        should_download = False
else:
    print(f"\n📥 Dataset não encontrado. Iniciando download...")
    should_download = True

# Executar download se necessário
if should_download:
    try:
        print(f"\n📥 Baixando dataset...")
        print(f"🌐 URL: {drive_url[:50]}...")
        print(f"📁 Destino: {dest_dir}")
        
        # Download do arquivo zip com barra de progresso
        response = requests.get(drive_url, stream=True)
        response.raise_for_status()
        
        total_size = int(response.headers.get('content-length', 0))
        downloaded_size = 0
        
        with open(zip_filename, 'wb') as f:
            for chunk in response.iter_content(chunk_size=8192):
                if chunk:
                    f.write(chunk)
                    downloaded_size += len(chunk)
                    
                    # Mostrar progresso simples
                    if total_size > 0:
                        progress = (downloaded_size / total_size) * 100
                        print(f"\r📊 Progresso: {progress:.1f}% ({downloaded_size/(1024*1024):.1f}MB)", end="")
        
        print(f"\n✅ Download concluído!")
        
        # Extrair o zip
        print(f"📦 Extraindo arquivos...")
        with zipfile.ZipFile(zip_filename, 'r') as zip_ref:
            # Extrai para uma pasta temporária
            temp_extract_dir = "temp_dataset"
            zip_ref.extractall(temp_extract_dir)
        
        # Criar pasta de destino se não existir
        os.makedirs(dest_dir, exist_ok=True)
        
        # Mover conteúdo extraído para a pasta de destino
        for item in os.listdir(temp_extract_dir):
            s = os.path.join(temp_extract_dir, item)
            d = os.path.join(dest_dir, item)
            if os.path.isdir(s):
                if os.path.exists(d):
                    shutil.rmtree(d)
                shutil.move(s, d)
            else:
                shutil.move(s, d)
        
        # Limpar arquivos temporários
        shutil.rmtree(temp_extract_dir)
        os.remove(zip_filename)
        
        print(f"✅ Dataset extraído com sucesso!")
        print(f"📁 Localização: {os.path.abspath(dest_dir)}")
        
    except Exception as e:
        print(f"❌ Erro durante o download: {e}")
        # Limpar arquivos em caso de erro
        if os.path.exists(zip_filename):
            os.remove(zip_filename)
        if os.path.exists("temp_dataset"):
            shutil.rmtree("temp_dataset")
else:
    print(f"\n🚀 Prosseguindo com dataset existente...")
    print(f"📁 Localização: {os.path.abspath(dest_dir)}")

print(f"\n🎯 PRÓXIMOS PASSOS:")
print(f"✅ Dataset verificado/baixado")
print(f"📊 Agora você pode prosseguir para o download dos modelos")

🔍 VERIFICANDO DATASET EXISTENTE...
📁 Pasta 'Data' encontrada com 1 subpasta(s):
   📂 fruit.v11i.yolov11 - 24157 arquivos
✅ Dataset de frutas completo encontrado!

❓ CONFIRMAÇÃO NECESSÁRIA:
Um dataset já foi encontrado na pasta 'Data'.

Opções:
  1️⃣ Pular download (usar dataset existente)
  2️⃣ Baixar novamente (sobrescrever)
  3️⃣ Baixar em nova pasta (manter ambos)
✅ Usando dataset existente!

🚀 Prosseguindo com dataset existente...
📁 Localização: c:\Users\Arklok\Documents\ProjetosPessoais\Classificao-de-Alimentos-YOLO\Data

🎯 PRÓXIMOS PASSOS:
✅ Dataset verificado/baixado
📊 Agora você pode prosseguir para o download dos modelos


# Donwload de Modelos para Treinamento

In [2]:
# Download de todos os modelos YOLOv11 disponíveis
import time

# Lista de todos os modelos YOLOv11 disponíveis
yolo_models = {
    'yolo11n.pt': 'Nano - Mais rápido, menor precisão (~6MB)',
    'yolo11s.pt': 'Small - Balanceado velocidade/precisão (~22MB)', 
    'yolo11m.pt': 'Medium - Boa precisão, velocidade moderada (~50MB)',
    'yolo11l.pt': 'Large - Alta precisão, mais lento (~52MB)',
    'yolo11x.pt': 'Extra Large - Máxima precisão, mais lento (~138MB)'
}

print("🤖 BAIXANDO TODOS OS MODELOS YOLOv11...")
print("=" * 60)

models_downloaded = []
models_dir = "Models"

# Criar diretório se não existir
os.makedirs(models_dir, exist_ok=True)

for model_name, description in yolo_models.items():
    print(f"\n📥 Baixando {model_name}...")
    print(f"   {description}")
    
    try:
        start_time = time.time()
        
        # Caminho completo do modelo
        model_path = os.path.join(models_dir, model_name)
        
        # Carregar modelo (isso fará o download se necessário)
        model = YOLO(model_path)
        
        download_time = time.time() - start_time
        
        # Verificar tamanho do arquivo
        if os.path.exists(model_path):
            file_size = os.path.getsize(model_path) / (1024*1024)  # MB
            print(f"   ✅ Sucesso! Tamanho: {file_size:.1f} MB")
            print(f"   ⏱️ Tempo: {download_time:.1f}s")
            models_downloaded.append(model_name)
        else:
            print(f"   ⚠️ Modelo pode estar no cache do ultralytics")
            models_downloaded.append(model_name)
            
    except Exception as e:
        print(f"   ❌ Erro ao baixar {model_name}: {e}")

print("\n" + "=" * 60)
print("📋 RESUMO DO DOWNLOAD:")
print(f"✅ Modelos baixados com sucesso: {len(models_downloaded)}")
for model in models_downloaded:
    print(f"   🤖 {model}")

print(f"\n📁 Localização: {os.path.abspath(models_dir)}")
print("🎯 Todos os modelos estão prontos para treinamento!")

🤖 BAIXANDO TODOS OS MODELOS YOLOv11...

📥 Baixando yolo11n.pt...
   Nano - Mais rápido, menor precisão (~6MB)
   ✅ Sucesso! Tamanho: 5.4 MB
   ⏱️ Tempo: 0.1s

📥 Baixando yolo11s.pt...
   Small - Balanceado velocidade/precisão (~22MB)
   ✅ Sucesso! Tamanho: 18.4 MB
   ⏱️ Tempo: 0.0s

📥 Baixando yolo11m.pt...
   Medium - Boa precisão, velocidade moderada (~50MB)
   ✅ Sucesso! Tamanho: 38.8 MB
   ⏱️ Tempo: 0.1s

📥 Baixando yolo11l.pt...
   Large - Alta precisão, mais lento (~52MB)
   ✅ Sucesso! Tamanho: 49.0 MB
   ⏱️ Tempo: 0.2s

📥 Baixando yolo11x.pt...
   Extra Large - Máxima precisão, mais lento (~138MB)
   ✅ Sucesso! Tamanho: 109.3 MB
   ⏱️ Tempo: 0.2s

📋 RESUMO DO DOWNLOAD:
✅ Modelos baixados com sucesso: 5
   🤖 yolo11n.pt
   🤖 yolo11s.pt
   🤖 yolo11m.pt
   🤖 yolo11l.pt
   🤖 yolo11x.pt

📁 Localização: c:\Users\Arklok\Documents\ProjetosPessoais\Classificao-de-Alimentos-YOLO\Models
🎯 Todos os modelos estão prontos para treinamento!


# Verificação do Ambiente e Dependências

In [3]:
# Verificar versões das bibliotecas importantes
import torch
import cv2
import numpy as np
import sys
from ultralytics import __version__ as ultralytics_version

print("=== INFORMAÇÕES DO AMBIENTE ===")
print(f"🐍 Python: {sys.version}")
print(f"🔥 PyTorch: {torch.__version__}")
print(f"👁️ OpenCV: {cv2.__version__}")
print(f"🔢 NumPy: {np.__version__}")
print(f"🚀 Ultralytics: {ultralytics_version}")

print("\n=== RECURSOS DE HARDWARE ===")
print(f"💻 CPU Cores: {os.cpu_count()}")
print(f"🎮 CUDA Disponível: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"🎮 GPU: {torch.cuda.get_device_name(0)}")
    print(f"🎮 CUDA Version: {torch.version.cuda}")
    print(f"🎮 Memória GPU: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")
else:
    print("⚠️ CUDA não disponível - treinamento será feito na CPU")

=== INFORMAÇÕES DO AMBIENTE ===
🐍 Python: 3.12.7 (tags/v3.12.7:0b05ead, Oct  1 2024, 03:06:41) [MSC v.1941 64 bit (AMD64)]
🔥 PyTorch: 2.8.0+cpu
👁️ OpenCV: 4.10.0
🔢 NumPy: 2.2.6
🚀 Ultralytics: 8.3.193

=== RECURSOS DE HARDWARE ===
💻 CPU Cores: 12
🎮 CUDA Disponível: False
⚠️ CUDA não disponível - treinamento será feito na CPU


## ANÁLISE DETALHADA DO DATASET

In [ ]:
# Análise detalhada do dataset de frutas
import yaml

print("=== ANÁLISE DETALHADA DO DATASET ===")

# Ler o arquivo data.yaml
data_yaml_path = r"Data/fruit.v11i.yolov11/data.yaml"
try:
    with open(data_yaml_path, 'r') as f:
        data_config = yaml.safe_load(f)
    
    print(f"📊 Número de classes: {data_config.get('nc', 'Não especificado')}")
    print(f"🏷️ Classes: {data_config.get('names', 'Não especificado')}")
    print(f"📁 Caminho de treino: {data_config.get('train', 'Não especificado')}")
    print(f"📁 Caminho de validação: {data_config.get('val', 'Não especificado')}")
    print(f"📁 Caminho de teste: {data_config.get('test', 'Não especificado')}")
    
    # Calcular estatísticas
    total_images = 10714 + 707 + 655  # train + valid + test
    print(f"\n📈 ESTATÍSTICAS:")
    print(f"  📸 Total de imagens: {total_images}")
    print(f"  🎯 Treino: {10714} ({10714/total_images*100:.1f}%)")
    print(f"  ✅ Validação: {707} ({707/total_images*100:.1f}%)")
    print(f"  🧪 Teste: {655} ({655/total_images*100:.1f}%)")
    
    print(f"\n🎯 DISTRIBUIÇÃO IDEAL:")
    print(f"  ✅ Proporção treino/validação: {10714/707:.1f}:1 (boa)")
    print(f"  ✅ Dataset bem balanceado para treinamento")
    
except Exception as e:
    print(f"❌ Erro ao ler data.yaml: {e}")

print(f"\n💾 ESPAÇO EM DISCO:")
# Estimar tamanho do dataset (aproximado)
estimated_size = total_images * 0.2  # Assumindo ~200KB por imagem
print(f"  📸 Tamanho estimado das imagens: ~{estimated_size/1024:.1f} GB")

print(f"\n⏱️ TEMPO ESTIMADO DE TREINAMENTO (CPU):")
print(f"  🕐 YOLOv11n: ~2-4 horas para 100 epochs")
print(f"  🕒 YOLOv11s: ~4-8 horas para 100 epochs")
print(f"  ⚠️ Recomendação: Use GPU para treinamento mais rápido")

=== ANÁLISE DETALHADA DO DATASET ===
📊 Número de classes: 7
🏷️ Classes: ['apple', 'banana', 'grape', 'guava', 'mango', 'orange', 'water melon']
📁 Caminho de treino: ../train/images
📁 Caminho de validação: ../valid/images
📁 Caminho de teste: ../test/images

📈 ESTATÍSTICAS:
  📸 Total de imagens: 12076
  🎯 Treino: 10714 (88.7%)
  ✅ Validação: 707 (5.9%)
  🧪 Teste: 655 (5.4%)

🎯 DISTRIBUIÇÃO IDEAL:
  ✅ Proporção treino/validação: 15.2:1 (boa)
  ✅ Dataset bem balanceado para treinamento

💾 ESPAÇO EM DISCO:
  📸 Tamanho estimado das imagens: ~2.4 GB

⏱️ TEMPO ESTIMADO DE TREINAMENTO (CPU):
  🕐 YOLOv11n: ~2-4 horas para 100 epochs
  🕒 YOLOv11s: ~4-8 horas para 100 epochs
  ⚠️ Recomendação: Use GPU para treinamento mais rápido


## VERIFICAÇÃO FINAL DOS MODELOS

In [ ]:
# Verificação final dos modelos baixados
print("🔍 VERIFICAÇÃO FINAL DOS MODELOS:")
print("=" * 50)

models_dir = "Models"
total_size = 0

if os.path.exists(models_dir):
    models = [f for f in os.listdir(models_dir) if f.endswith('.pt')]
    
    if models:
        print(f"📁 Diretório: {os.path.abspath(models_dir)}")
        print(f"📊 Total de modelos: {len(models)}")
        print()
        
        for model in sorted(models):
            model_path = os.path.join(models_dir, model)
            size_mb = os.path.getsize(model_path) / (1024*1024)
            total_size += size_mb
            
            # Emoji para cada tipo de modelo
            emoji = "⚡" if "n" in model else "🔥" if "s" in model else "💪" if "m" in model else "🚀" if "l" in model else "🦾"
            
            print(f"{emoji} {model:<12} - {size_mb:>6.1f} MB")
        
        print("-" * 30)
        print(f"💾 TOTAL: {total_size:.1f} MB ({total_size/1024:.2f} GB)")
        
        print(f"\n🎯 RECOMENDAÇÕES DE USO:")
        print(f"🏃‍♂️ Para teste rápido: yolo11n.pt")
        print(f"⚖️ Para balanceado: yolo11s.pt ou yolo11m.pt") 
        print(f"🎯 Para máxima precisão: yolo11l.pt ou yolo11x.pt")
        
    else:
        print("❌ Nenhum modelo encontrado na pasta!")
else:
    print("❌ Pasta de modelos não encontrada!")

🔍 VERIFICAÇÃO FINAL DOS MODELOS:
📁 Diretório: c:\Users\Arklok\Documents\ProjetosPessoais\Classificao-de-Alimentos-YOLO\Models
📊 Total de modelos: 5

🚀 yolo11l.pt   -   49.0 MB
💪 yolo11m.pt   -   38.8 MB
⚡ yolo11n.pt   -    5.4 MB
🔥 yolo11s.pt   -   18.4 MB
🚀 yolo11x.pt   -  109.3 MB
------------------------------
💾 TOTAL: 220.9 MB (0.22 GB)

🎯 RECOMENDAÇÕES DE USO:
🏃‍♂️ Para teste rápido: yolo11n.pt
⚖️ Para balanceado: yolo11s.pt ou yolo11m.pt
🎯 Para máxima precisão: yolo11l.pt ou yolo11x.pt
💡 Para seu dataset: recomendo começar com yolo11s.pt


# Treinamento do Modelo

In [ ]:
# Verificação automática de dispositivo disponível
import torch

print("🔍 VERIFICANDO DISPOSITIVOS DISPONÍVEIS...")
if torch.cuda.is_available():
    device = '0'  # GPU
    device_name = torch.cuda.get_device_name(0)
    print(f"🎮 Usando GPU: {device_name}")
else:
    device = 'cpu'  # CPU
    print(f"💻 Usando CPU: {os.cpu_count()} cores")

print(f"🎯 Dispositivo selecionado: {device}")

# Carregando um modelo pré-treinado para treinamento
path_model = r"Models/yolo11n.pt"
dataset_path = r"Data/fruit.v11i.yolov11/data.yaml"

print(f"\n🤖 Carregando modelo: {path_model}")
print(f"📊 Dataset: {dataset_path}")

model = YOLO(path_model)

# Configurações de treinamento ajustadas para CPU
print(f"\n🚀 INICIANDO TREINAMENTO...")
print(f"  📏 Tamanho da imagem: 640px")
print(f"  💻 Dispositivo: {device}")

# Treinamento do Modelo com configurações otimizadas para CPU
model_train = model.train(
    data=dataset_path, 
    epochs=5, 
    batch=0.80,  # Reduzido para CPU
    imgsz=640, 
    name="yolo11n-fruit", 
    exist_ok=True, 
    device=device,  # Usa dispositivo detectado automaticamente
    workers=os.cpu_count(),  # Número de workers para CPU
    patience=50,  # Paciência para early stopping
    save=True,  # Salvar checkpoints
    plots=True  # Gerar gráficos de treinamento
    
)

print("\n✅ TREINAMENTO Concluído!")

# Validação do Treinamento do Modelo

In [3]:
# Load a model
model = YOLO(r"runs\weights\best.pt")  # load a custom model

# Validate the model
metrics = model.val()  # no arguments needed, dataset and settings remembered
metrics.box.map  # map50-95
metrics.box.map50  # map50
metrics.box.map75  # map75
metrics.box.maps  # a list contains map50-95 of each category

Ultralytics 8.3.193  Python-3.12.7 torch-2.8.0+cpu CPU (12th Gen Intel Core(TM) i5-1235U)
YOLO11n summary (fused): 100 layers, 2,583,517 parameters, 0 gradients, 6.3 GFLOPs
val: Fast image access  (ping: 0.20.2 ms, read: 52.513.0 MB/s, size: 55.9 KB)
val: Scanning C:\Users\Arklok\Documents\ProjetosPessoais\Classificao-de-Alimentos-YOLO\Data\fruit.v11i.yolov11\valid\labels.cache... 707 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 707/707  0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 45/45 0.4it/s 1:412.2sss
                   all        707       2089      0.608      0.586      0.624      0.488
                 apple         95        355      0.874       0.27      0.659      0.528
                banana          2          6     0.0342      0.333     0.0185    0.00873
                 grape        112        250      0.894      0.644      0.827      0.609
                 guava        113        270      0.492  

array([     0.5283,   0.0087297,     0.60929,     0.47601,      0.5568,     0.60499,      0.6285])

# Testando o Modelo Final

In [19]:
INPUT_VAL_FOLDER = r"C:\Users\Arklok\Documents\ProjetosPessoais\Classificao-de-Alimentos-YOLO\DataProd\Input"
OUTPUT_VAL_FOLDER = r"C:\Users\Arklok\Documents\ProjetosPessoais\Classificao-de-Alimentos-YOLO\DataProd\Output"

v_it = int(len(os.listdir(OUTPUT_VAL_FOLDER))) + 1
OUTPUT_VAL_FOLDER = os.path.join(OUTPUT_VAL_FOLDER, f"{v_it}")
os.makedirs(OUTPUT_VAL_FOLDER, exist_ok=True)

In [20]:
# Certifique-se de carregar o modelo corretamente antes de prever
model = YOLO(r"runs\weights\best.pt")  # load a custom model
model_obj = YOLO(model)  # 'model' é o caminho do modelo salvo

In [21]:
for file in os.listdir(INPUT_VAL_FOLDER):
    if file.lower().endswith(('.png', '.jpg', '.jpeg', '.bmp', '.tiff')):
        input_image_path = os.path.join(INPUT_VAL_FOLDER, file)
        output_image_path = os.path.join(OUTPUT_VAL_FOLDER, file)
        
        print(f"\n📥 Processando imagem: {input_image_path}")
        
        results = model_obj.predict(
            source=input_image_path
        )
        
        print(f"📤 Salvando resultado de {file}!")
        results[0].save(filename=output_image_path)


📥 Processando imagem: C:\Users\Arklok\Documents\ProjetosPessoais\Classificao-de-Alimentos-YOLO\DataProd\Input\811a6dea-cad8-4698-bb62-bc0b575eac86.jpeg

image 1/1 C:\Users\Arklok\Documents\ProjetosPessoais\Classificao-de-Alimentos-YOLO\DataProd\Input\811a6dea-cad8-4698-bb62-bc0b575eac86.jpeg: 640x480 4 guavas, 101.7ms
Speed: 2.5ms preprocess, 101.7ms inference, 1.7ms postprocess per image at shape (1, 3, 640, 480)
📤 Salvando resultado de 811a6dea-cad8-4698-bb62-bc0b575eac86.jpeg!

📥 Processando imagem: C:\Users\Arklok\Documents\ProjetosPessoais\Classificao-de-Alimentos-YOLO\DataProd\Input\maça.png

image 1/1 C:\Users\Arklok\Documents\ProjetosPessoais\Classificao-de-Alimentos-YOLO\DataProd\Input\maa.png: 640x640 9 oranges, 178.5ms
Speed: 5.2ms preprocess, 178.5ms inference, 3.1ms postprocess per image at shape (1, 3, 640, 640)
📤 Salvando resultado de maça.png!

📥 Processando imagem: C:\Users\Arklok\Documents\ProjetosPessoais\Classificao-de-Alimentos-YOLO\DataProd\Input\Mídia (1).jpeg

